# 00 — Download the CB2 data

This notebook downloads the current public ChEMBL data for **human CB2 (CHEMBL253, UniProt P34972)**. Run the cells from top to bottom with a Python 3 kernel and an internet connection. Only Python's built-in libraries are needed.

We keep the rough draft's scope: all activity measurements linked to the target, their assay metadata, and documents referenced by those measurements. **No Ki, organism, assay-quality, or duplicate filters are applied here.** Those decisions belong in exploration.

Compared with the draft, this version separates each task into a short explained step and uses ordinary lists and loops instead of pandas. It preserves pagination, raw field values, identifier checks, database-release checks, and acquisition metadata.

Each run creates a new dated folder under this repository's `data/raw`. Existing downloads are never overwritten. A folder is complete only when it contains `acquisition_manifest.json`, which is written after validation.

API reference: [ChEMBL Data Web Services](https://chembl.gitbook.io/chembl-interface-documentation/web-services/chembl-data-web-services).

## 1. Choose the target and output location

`Path` handles file locations. The working directory can be either the repository root or its `notebooks` folder. The checks below prevent accidentally saving into another project.

UTC gives the download a consistent timestamp regardless of local time zone.

In [ ]:
# Import the built-in Python tools used for file paths, timestamps, web requests, URL building, and JSON.
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import urlopen
from urllib.parse import urlencode, urljoin
import json

# Base address for the ChEMBL REST API and the ChEMBL ID for the human CB2 receptor.
BASE_URL = "https://www.ebi.ac.uk/chembl/api/data/"
TARGET_ID = "CHEMBL253"

# Find the repository root whether this notebook is run from the repo itself or from its notebooks folder.
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

# Safeguards: stop if this does not look like the expected project folder.
assert (project_root / "notebooks").is_dir(), "Run from the repository or notebooks folder."
assert (project_root / "data" / "raw").is_dir(), "Expected this repository's data/raw folder."

# Timestamp this download and use it to create a new, uniquely named raw-data folder.
retrieved_at = datetime.now(timezone.utc)
folder_name = "chembl_cb2_" + retrieved_at.strftime("%Y%m%dT%H%M%S%fZ")
download_folder = project_root / "data" / "raw" / folder_name

# Show exactly where this acquisition will be saved.
print("Repository:", project_root)
print("Download folder:", download_folder)

Repository: /home/justin/code/cb2-affinity
Download folder: /home/justin/code/cb2-affinity/data/raw/chembl_cb2_20260922T165756862384Z


## 2. Read JSON and confirm the target

The API returns JSON: dictionaries for named fields and lists for collections of records. These two small helpers read a response and save it without changing its field values.

Before downloading measurements, confirm that the target is human CB2 and record the current ChEMBL release. A request failure stops the notebook rather than silently skipping data.

In [ ]:
# Helper function: request a ChEMBL URL and convert its JSON response into Python objects.
def get_json(url):
    with urlopen(url, timeout=90) as response:
        return json.load(response)


# Helper function: save Python data as readable JSON without changing the underlying values.
def save_json(filename, records):
    path = download_folder / filename
    with path.open("w", encoding="utf-8") as file:
        json.dump(records, file, indent=2, ensure_ascii=False)


# Ask ChEMBL for the current database release and for the target record identified by CHEMBL253.
status = get_json(BASE_URL + "status.json")
target = get_json(BASE_URL + "target/" + TARGET_ID + ".json")

# Safeguards: confirm that CHEMBL253 is the human CB2 target we intend to study.
assert target["target_chembl_id"] == TARGET_ID
assert target["organism"] == "Homo sapiens"
# Collect the UniProt accession numbers attached to the target and verify the expected CB2 accession.
accessions = []
for component in target["target_components"]:
    accessions.append(component["accession"])
assert "P34972" in accessions, "Expected the CB2 UniProt accession."

# Only after the target checks pass, create the output folder and save the target/release metadata.
download_folder.mkdir(parents=True, exist_ok=False)
save_json("chembl_status.json", status)
save_json("chembl_cb2_target.json", target)

# Print a short human-readable confirmation of what target and database release were retrieved.
print("ChEMBL release:", status["chembl_db_version"])
print("Target:", target["pref_name"])
print("Organism:", target["organism"])

ChEMBL release: ChEMBL_37
Target: Cannabinoid receptor 2
Organism: Homo sapiens


## 3. Download every page of activity measurements

ChEMBL divides large results into pages. Downloading only the first page would miss measurements. The loop adds each page's records to a list and follows `next` until there are no pages left. We check the final count against the API's expected total.

The only query filter is the target ID. `limit` controls page size, not the total download size.

In [ ]:
# Download all pages from a ChEMBL endpoint and verify that the full result set was collected.
def download_collection(first_url, records_key):
    records = []
    url = first_url
    expected_total = None

    while url:
        page = get_json(url)

        # ChEMBL reports the total number of matching records in page_meta.
        # Save it from the first page and make sure it stays consistent during pagination.
        if expected_total is None:
            expected_total = page["page_meta"]["total_count"]

        assert page["page_meta"]["total_count"] == expected_total, "API count changed during download."

        # Add the current page of records to the growing dataset.
        records.extend(page[records_key])
        print(f"{records_key}: {len(records):,} / {expected_total:,}")

        # Follow ChEMBL's "next" link until there are no more pages.
        next_page = page["page_meta"]["next"]
        if next_page:
            url = urljoin(BASE_URL, next_page)
        else:
            url = None

    # Confirm that we actually downloaded the number of records ChEMBL reported.
    assert len(records) == expected_total, "The download is incomplete."
    return records


# Request all activity records associated with the human CB2 target.
# limit=1000 controls the number of records returned per API page.
activity_url = BASE_URL + "activity.json?" + urlencode({
    "target_chembl_id": TARGET_ID,
    "limit": 1000,
})

activity_records = download_collection(activity_url, "activities")

# Preserve the original API response as raw data for later exploration and cleaning.
save_json("chembl_cb2_activity.json", activity_records)

activities: 1,000 / 22,523
activities: 2,000 / 22,523
activities: 3,000 / 22,523
activities: 4,000 / 22,523
activities: 5,000 / 22,523
activities: 6,000 / 22,523
activities: 7,000 / 22,523
activities: 8,000 / 22,523
activities: 9,000 / 22,523
activities: 10,000 / 22,523
activities: 11,000 / 22,523
activities: 12,000 / 22,523
activities: 13,000 / 22,523
activities: 14,000 / 22,523
activities: 15,000 / 22,523
activities: 16,000 / 22,523
activities: 17,000 / 22,523
activities: 18,000 / 22,523
activities: 19,000 / 22,523
activities: 20,000 / 22,523
activities: 21,000 / 22,523
activities: 22,000 / 22,523
activities: 22,523 / 22,523


## 4. Download assay metadata

An activity is a measurement; an assay describes the experiment that produced it. Keep the full assay records so exploration can inspect descriptions, organisms, and target confidence.

In [ ]:
# Download and save all assay metadata for the same human CB2 target.
# These assay records help us interpret the experimental context behind each activity measurement.

assay_url = BASE_URL + "assay.json?" + urlencode({
    "target_chembl_id": TARGET_ID,
    "limit": 1000,
})

assay_records = download_collection(assay_url, "assays")
save_json("chembl_cb2_assay.json", assay_records)

assays: 1,000 / 1,551
assays: 1,551 / 1,551


## 5. Download source documents

Collect the document IDs referenced by the activities. A set keeps each ID once; sorting makes the request order repeatable. Missing document IDs are left as they are in the raw activities.

Request 50 IDs at a time to keep URLs manageable. `__in` asks ChEMBL for any of the listed IDs. Each batch still uses pagination.

In [ ]:
# Collect the unique ChEMBL document IDs referenced by the activity data.
# Sorting keeps the request order consistent between runs.

document_ids = set()
for activity in activity_records:
    document_id = activity.get("document_chembl_id")
    if document_id:
        document_ids.add(document_id)

document_ids = sorted(document_ids)


# Download document metadata in batches of 50 IDs.
# ChEMBL's "__in" filter lets one request retrieve any document matching the IDs in that batch.
document_records = []
document_urls = []

for start in range(0, len(document_ids), 50):
    batch = document_ids[start:start + 50]

    url = BASE_URL + "document.json?" + urlencode({
        "document_chembl_id__in": ",".join(batch),
        "limit": 1000,
    })

    # Save the request URLs for provenance while combining all returned documents into one dataset.
    document_urls.append(url)
    batch_records = download_collection(url, "documents")
    document_records.extend(batch_records)


save_json("chembl_cb2_documents.json", document_records)
print("Documents downloaded:", len(document_records))

documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 50 / 50
documents: 3 / 3
Documents downloaded: 653


## 6. Check completeness and table links

Check that IDs are present and unique, every activity's assay is available, and all requested documents arrived. These checks validate the download; they do not remove records or judge measurement quality.

Also confirm that ChEMBL did not switch database releases during the run.

In [ ]:
# Extract the primary IDs from each downloaded dataset so we can verify that
# records are complete, unique, and correctly linked to one another.
activity_ids = [record["activity_id"] for record in activity_records]
assay_ids = [record["assay_chembl_id"] for record in assay_records]
returned_document_ids = [record["document_chembl_id"] for record in document_records]


# Confirm that every record has an ID and that no primary IDs were duplicated.
for ids in [activity_ids, assay_ids, returned_document_ids]:
    assert all(ids), "A record is missing its ID."
    assert len(ids) == len(set(ids)), "Repeated record IDs need investigation."


# Verify that every activity is linked to a downloaded assay
# and still belongs to the intended human CB2 target.
available_assays = set(assay_ids)

for activity in activity_records:
    assert activity["assay_chembl_id"] in available_assays, "Missing assay metadata."
    assert activity["target_chembl_id"] == TARGET_ID


# Confirm that the documents we downloaded exactly match those referenced by the activities.
assert set(returned_document_ids) == set(document_ids), "Document IDs do not match."
assert len(activity_records) > 0, "No activities were returned."


# Check the database version again to make sure ChEMBL did not change releases
# while the acquisition was running.
status_after = get_json(BASE_URL + "status.json")
assert status_after["chembl_db_version"] == status["chembl_db_version"], "ChEMBL release changed; rerun."

print("All download checks passed.")

All download checks passed.


## 7. Save the acquisition record and verify the files

The manifest records when and what we downloaded, the release, exact queries, and counts. Writing it last marks a successfully checked download.

The filenames match the rough draft so the exploration notebook can use the same raw tables. The final check reads each saved table back from disk.

In [ ]:
# Count the distinct ChEMBL molecule IDs represented across all downloaded activity measurements.
molecule_ids = set()
for activity in activity_records:
    molecule_id = activity.get("molecule_chembl_id")
    if molecule_id:
        molecule_ids.add(molecule_id)

# Check the files on disk before marking this acquisition complete.
for filename, records in [
    ("chembl_cb2_activity.json", activity_records),
    ("chembl_cb2_assay.json", assay_records),
    ("chembl_cb2_documents.json", document_records),
]:
    with (download_folder / filename).open(encoding="utf-8") as file:
        saved_records = json.load(file)
    assert saved_records == records, "Saved records differ from the download."

# Build a manifest describing when, where, and how this raw acquisition was collected.
manifest = {
    "source": "ChEMBL REST API",
    "retrieved_at_utc": retrieved_at.isoformat(),
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "chembl_version": status["chembl_db_version"],
    "target_chembl_id": TARGET_ID,
    "target_name": target["pref_name"],
    "organism": target["organism"],
    "uniprot": "P34972",
    "activity_query": activity_url,
    "assay_query": assay_url,
    "document_queries": document_urls,
    "activity_records": len(activity_records),
    "distinct_molecule_ids": len(molecule_ids),
    "assay_records": len(assay_records),
    "document_records": len(document_records),
    "filtering_applied": "Target query only; no activity quality filters",
}
# Save the manifest only after the raw files have passed the integrity checks above.
save_json("acquisition_manifest.json", manifest)

# Print a final summary of the completed acquisition and list the files written to disk.
print(f"Activities: {len(activity_records):,}")
print(f"Distinct molecule IDs: {len(molecule_ids):,}")
print(f"Assays: {len(assay_records):,}")
print(f"Documents: {len(document_records):,}")
print("Saved and verified in:", download_folder)
for path in sorted(download_folder.iterdir()):
    print(" -", path.name, f"({path.stat().st_size:,} bytes)")

Activities: 22,523
Distinct molecule IDs: 11,399
Assays: 1,551
Documents: 653
Saved and verified in: /home/justin/code/cb2-affinity/data/raw/chembl_cb2_20260922T165756862384Z
 - acquisition_manifest.json (12,471 bytes)
 - chembl_cb2_activity.json (47,445,016 bytes)
 - chembl_cb2_assay.json (1,891,087 bytes)
 - chembl_cb2_documents.json (1,079,515 bytes)
 - chembl_cb2_target.json (8,802 bytes)
 - chembl_status.json (230 bytes)


## Stop here for review

The raw download is ready. No measurement cleaning or molecular processing has been performed. The next notebook will inspect these records, select human CB2 binding Ki measurements, and explain the cleaning choices one at a time.